In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt

df = pd.read_csv('assuming_independent_initial_synthetic.csv')

print("\n Data column information:") 
print(df.info())

# Check for missing values
print("\n The number of missing values in each column:")
print(df.isnull().sum())

# Convert income from continuous variable to categorical variable
print("\n Income descriptive statistics:")

df['income_level'] = pd.qcut(df['median_income'], q=5, labels=['very_low_income', 'low_income', 'medium_income', 'high_income', 'very_high_income'])

print(pd.qcut(df['median_income'], q=5).unique())

# Print the first few rows of the new binned column to confirm the operation was successful
print("\n The first 5 rows of data after binning:")
print(df.head())
print("\n The number of samples in each income level after binning:")
print(df['income_level'].value_counts())

features = ['gender', 'Age', 'Arrival_group', 'Business_size', 'Industry', 'Institutional_sector', 'Job_duration', 'Visa_group']
target = 'income_level'

# Remove missing values
df = df[features + [target]].dropna()

# Convert categorical variables to numbers (gender -> 0, 1)
le = LabelEncoder()
for col in features + [target]:
    df[col] = le.fit_transform(df[col])

# Training and testing
X = df[features]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


 Data column information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98993 entries, 0 to 98992
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   gender                98993 non-null  object 
 1   median_income         98993 non-null  float64
 2   Age                   98993 non-null  object 
 3   Arrival_group         98993 non-null  object 
 4   Business_size         98993 non-null  object 
 5   Industry              98991 non-null  object 
 6   Institutional_sector  98993 non-null  object 
 7   Job_duration          98993 non-null  object 
 8   Visa_group            98993 non-null  object 
dtypes: float64(1), object(8)
memory usage: 6.8+ MB
None

 The number of missing values in each column:
gender                  0
median_income           0
Age                     0
Arrival_group           0
Business_size           0
Industry                2
Institutional_sector    0
Job_duration      

In [2]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Testing dataset
y_pred_rf = rf.predict(X_test)
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print(f"Accuracy of the random forest model: {accuracy_rf:.2f}")

print("\n Random Forest Classification Report:")
print(classification_report(y_test, y_pred_rf))

# Feature Importance
feature_importances = pd.Series(rf.feature_importances_, index=X.columns)
sorted_importances = feature_importances.sort_values(ascending=False)

print("\n Features sorted by importance:")
print(sorted_importances)

Accuracy of the random forest model: 0.96

 Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.97      0.94      5983
           1       0.99      0.92      0.95      5854
           2       0.98      0.96      0.97      5956
           3       0.98      0.94      0.96      5941
           4       0.93      0.99      0.96      5964

    accuracy                           0.96     29698
   macro avg       0.96      0.96      0.96     29698
weighted avg       0.96      0.96      0.96     29698


 Features sorted by importance:
Age                     0.207447
Industry                0.187997
Job_duration            0.168295
Institutional_sector    0.140164
Visa_group              0.112606
Arrival_group           0.102541
Business_size           0.062281
gender                  0.018670
dtype: float64


In [7]:
import pandas as pd
import itertools
import numpy as np
from sklearn.model_selection import KFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score

# Prepare the data
df = pd.read_csv('assuming_independent_initial_synthetic.csv')

# Categorize median_income into 5 levels
df['income_level'] = pd.qcut(df['median_income'], q=5, labels=['very_low_income', 'low_income', 'medium_income', 'high_income', 'very_high_income'])

# Create binary target variable: 0 for high income, 1 for not high income
df['income_group'] = df['income_level'].replace({
    'very_low_income': 'not_high_income',
    'low_income': 'not_high_income',
    'medium_income': 'not_high_income',
    'high_income': 'high_income',
    'very_high_income': 'high_income'
})

all_features = ['gender', 'Age', 'Arrival_group', 'Business_size', 'Industry', 'Institutional_sector', 'Job_duration', 'Visa_group']
target = 'income_group'

# Delete rows with missing values in features or target
df = df[all_features + [target]].dropna()

# Encode categorical variables
le_target = LabelEncoder()
df[target] = le_target.fit_transform(df[target])

# Encode all features
for col in all_features:
    le_features = LabelEncoder()
    df[col] = le_features.fit_transform(df[col])

# Define positive class label
POS_LABEL = 0 if le_target.classes_[0] == 'high_income' else 1 

# Configure cross-validation (K-Fold)
K_FOLDS = 5 
cv = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42) # Fixed split, but model training is random each time

# Define scoring metrics
scoring = {
    'recall': make_scorer(recall_score, average='binary', pos_label=POS_LABEL, zero_division=0),
    'f1_score': make_scorer(f1_score, average='binary', pos_label=POS_LABEL, zero_division=0),
    'precision': make_scorer(precision_score, average='binary', pos_label=POS_LABEL, zero_division=0),
    'accuracy': make_scorer(accuracy_score)
}

# Store results
results = []

# Iteration over all feature combinations with cross-validation

for r in range(1, len(all_features) + 1):
    for combo in itertools.combinations(all_features, r):
        try:
            X = df[list(combo)]
            y = df[target]

            # Don't set random_state when creating the model each time, let it be naturally random in each cross-validation
            model = RandomForestClassifier(n_estimators=100) 

            # K-fold cross-validation
            scores = cross_validate(model, X, y, cv=cv, scoring=scoring, n_jobs=-1)

            # Calculate average scores
            avg_rec = np.mean(scores['test_recall'])
            avg_f1 = np.mean(scores['test_f1_score'])
            avg_prec = np.mean(scores['test_precision'])
            avg_acc = np.mean(scores['test_accuracy'])

            results.append({
                'combo': list(combo),
                'recall': avg_rec,
                'f1_score': avg_f1,
                'precision': avg_prec,
                'accuracy': avg_acc
            })
            
        except Exception as e:
            # Record error combination with zero scores
            results.append({
                'combo': list(combo), 'accuracy': 0.0, 'precision': 0.0, 'recall': 0.0, 'f1_score': 0.0
            })
            print(f"combination {combo} error: {e}")


# Sort by F1-Score and print top 10
top_10 = sorted(results, key=lambda x: x['f1_score'], reverse=True)[:10]

print("\nTop 10 feature combinations that have the most significant impact on income (sorted by average F1-Score):")
print(f"{'Rank':<4} |  {'Recall':<8} | {'F1-Score':<8} | {'Precision':<10} | {'Accuracy':<8} | {'Feature Combination'}")
for i, res in enumerate(top_10, 1):
    print(
        f"{i:<4} | "
        f"{res['recall']:.4f} | "
        f"{res['f1_score']:.4f} | "
        f"{res['precision']:.4f} | "
        f"{res['accuracy']:.4f} | "
        f"{res['combo']}"
    )

/var/folders/ml/19v89_p50w3_7k8ml02v9s100000gn/T/ipykernel_87595/1315358707.py:16: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df['income_group'] = df['income_level'].replace({



Top 10 feature combinations that have the most significant impact on income (sorted by average F1-Score):
Rank |  Recall   | F1-Score | Precision  | Accuracy | Feature Combination
1    | 0.9959 | 0.9857 | 0.9756 | 0.9884 | ['Age', 'Arrival_group', 'Business_size', 'Industry', 'Institutional_sector', 'Job_duration', 'Visa_group']
2    | 0.9960 | 0.9857 | 0.9756 | 0.9884 | ['Age', 'Arrival_group', 'Business_size', 'Industry', 'Institutional_sector', 'Visa_group']
3    | 0.9947 | 0.9857 | 0.9768 | 0.9884 | ['gender', 'Age', 'Arrival_group', 'Business_size', 'Industry', 'Institutional_sector', 'Job_duration', 'Visa_group']
4    | 0.9956 | 0.9856 | 0.9759 | 0.9884 | ['Age', 'Arrival_group', 'Industry', 'Institutional_sector', 'Visa_group']
5    | 0.9956 | 0.9856 | 0.9759 | 0.9884 | ['Age', 'Arrival_group', 'Industry', 'Institutional_sector', 'Job_duration', 'Visa_group']
6    | 0.9948 | 0.9856 | 0.9767 | 0.9884 | ['gender', 'Age', 'Arrival_group', 'Business_size', 'Industry', 'Institutiona